In [2]:
!pip install mtcnn opencv-python-headless

In [3]:
import os
import cv2
from mtcnn import MTCNN
from tqdm import tqdm

2026-05-06 11:36:32.939025: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778067393.125123      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778067393.179345      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778067393.640374      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778067393.640419      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778067393.640422      58 computation_placer.cc:177] computation placer alr

In [4]:
INPUT_PATH = "/kaggle/input/datasets/kshitizbhargava/deepfake-face-images/Final Dataset"
OUTPUT_PATH = "/kaggle/working/face_dataset"

classes = ["Fake", "Real"]

# Create output folders
for cls in classes:
    os.makedirs(os.path.join(OUTPUT_PATH, cls), exist_ok=True)

print("✅ Output folders created")

✅ Output folders created


In [5]:
detector = MTCNN()

I0000 00:00:1778067412.769401      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778067412.775493      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [ ]:
IMG_SIZE = 224

for cls in classes:
    input_folder = os.path.join(INPUT_PATH, cls)
    output_folder = os.path.join(OUTPUT_PATH, cls)

    images = os.listdir(input_folder)

    print(f"\n🔄 Processing {cls} ({len(images)} images)")

    for img_name in tqdm(images):
        img_path = os.path.join(input_folder, img_name)

        image = cv2.imread(img_path)
        if image is None:
            continue

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        results = detector.detect_faces(image)

        if len(results) == 0:
            continue  # skip if no face

        x, y, w, h = results[0]['box']
        x, y = max(0, x), max(0, y)

        face = image[y:y+h, x:x+w]

        try:
            face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
        except:
            continue

        save_path = os.path.join(output_folder, img_name)

        cv2.imwrite(save_path, cv2.cvtColor(face, cv2.COLOR_RGB2BGR))

print("\n✅ Face extraction completed!")


🔄 Processing Fake (7000 images)


 23%|██▎       | 1617/7000 [04:34<14:57,  6.00it/s]

In [ ]:
for cls in classes:
    count = len(os.listdir(os.path.join(OUTPUT_PATH, cls)))
    print(f"{cls}: {count} images")

In [ ]:
import shutil

shutil.make_archive("face_dataset", 'zip', OUTPUT_PATH)

print("✅ Zipped dataset ready for download")